# TP : Apprentissage par Renforcement

**Trois algorithmes couverts :**

| Algo | Famille | Environnement | Caractéristique |
|---|---|---|---|
| Q-Learning | Tabulaire | CartPole-v1 | Baseline classique, espace discret |
| DQN | Deep RL (value-based) | CartPole-v1 | Q-Learning + réseau de neurones |
| PPO | Deep RL (policy-based) | MountainCar-v0 | State-of-the-art, robuste |

**Objectifs :**
- Comprendre la boucle agent-environnement
- Implémenter Q-Learning from scratch
- Utiliser Stable-Baselines3 pour DQN et PPO
- Évaluer et comparer les politiques apprises

**Prérequis :**
Voir `pip install -r requirements.txt` a la racine de `cours_ml/todo/`.

> **Version exercice** : les cellules marquees `# TODO` sont a completer. Le reste (imports, chargement des donnees, affichages) est deja fourni.
> Installe les dependances une seule fois avec `pip install -r requirements.txt` depuis `cours_ml/todo/` (voir le README de ce dossier). Le corrige complet se trouve dans `cours_ml/03_renforcement/tp_reinforcement.ipynb`.


---
## 0. Imports & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print(f"gymnasium : {gym.__version__}")
print(f"stable-baselines3 : {__import__('stable_baselines3').__version__}")
print(f"torch : {torch.__version__}")
print(f"GPU disponible : {torch.cuda.is_available()}")

---
## 1. Rappels : la boucle agent-environnement

En apprentissage par renforcement, un **agent** interagit avec un **environnement** selon ce cycle :

```
état (s) → agent → action (a) → environnement → récompense (r) + nouvel état (s')
```

L'objectif est de maximiser la **récompense cumulée** (return) :
$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \ldots = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$$

où $\gamma \in [0, 1]$ est le **facteur de discount** (importance du futur).

### Les deux environnements

**Datamap : variables d'état (observation)**

| Environnement | Variable | Description |
|---|---|---|
| CartPole-v1 | Position du chariot | Position horizontale du chariot sur le rail |
| CartPole-v1 | Vitesse du chariot | Vitesse horizontale du chariot |
| CartPole-v1 | Angle du pendule | Angle du pendule par rapport à la verticale |
| CartPole-v1 | Vitesse angulaire | Vitesse de rotation du pendule |
| MountainCar-v0 | Position | Position horizontale de la voiture sur la pente |
| MountainCar-v0 | Vitesse | Vitesse horizontale de la voiture |

In [ ]:
# CartPole-v1 : maintenir un pendule en équilibre
env_cp = gym.make('CartPole-v1')
print("=== CartPole-v1 ===")
print(f"Espace d'observation : {env_cp.observation_space}  (4 valeurs continues)")
print(f"Espace d'actions     : {env_cp.action_space}  (0=gauche, 1=droite)")
print(f"Récompense           : +1 à chaque step tant que le pendule tient")
print(f"Condition d'arrêt    : pendule > 12°, chariot hors limites, ou 500 steps")
env_cp.close()

print()

# MountainCar-v0 : voiture qui doit atteindre le sommet
env_mc = gym.make('MountainCar-v0')
print("=== MountainCar-v0 ===")
print(f"Espace d'observation : {env_mc.observation_space}  (position, vitesse)")
print(f"Espace d'actions     : {env_mc.action_space}  (0=gauche, 1=neutre, 2=droite)")
print(f"Récompense           : -1 à chaque step (pénalité de temps)")
print(f"Condition d'arrêt    : position >= 0.5 (sommet) ou 200 steps")
env_mc.close()

---
## 2. Q-Learning (tabulaire) : CartPole-v1

Q-Learning apprend une **table Q(s, a)** : la valeur espérée de prendre l'action `a` dans l'état `s`.

Mise à jour de Bellman :
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

**Problème :** CartPole a un espace d'états continu → on **discrétise** les observations.

**Exploration vs Exploitation :** politique $\epsilon$-greedy : avec probabilité $\epsilon$, action aléatoire ; sinon, meilleure action connue. $\epsilon$ décroît au fil de l'entraînement.

### 2.1 Discrétisation de l'espace d'états

In [ ]:
# On découpe chaque dimension en N_BINS intervalles
N_BINS = 20

# Bornes manuelles (les bornes gym sont +/-inf pour certaines dims)
BINS = [
    np.linspace(-2.4,  2.4,  N_BINS),   # position chariot
    np.linspace(-3.0,  3.0,  N_BINS),   # vitesse chariot
    np.linspace(-0.21, 0.21, N_BINS),   # angle pendule
    np.linspace(-3.0,  3.0,  N_BINS),   # vitesse angulaire
]

def discretize(obs):
    """Convertit une observation continue en tuple d'indices discrets."""
    # TODO : pour chaque dimension i, trouver l'indice du bin correspondant a obs[i]
    # Indice : np.digitize(obs[i], BINS[i]) renvoie l'indice du bin
    ...

# Test
env = gym.make('CartPole-v1')
obs, _ = env.reset(seed=RANDOM_STATE)
print("Observation brute   :", obs)
print("Observation discrète:", discretize(obs))
env.close()


### 2.2 Entraînement Q-Learning

In [ ]:
# Hyperparamètres
N_EPISODES    = 5000
ALPHA         = 0.1    # taux d'apprentissage
GAMMA         = 0.99   # discount
EPSILON_START = 1.0
EPSILON_MIN   = 0.01
EPSILON_DECAY = 0.995

# Table Q initialisée à zéro
q_table = {}

def get_q(state, action):
    return q_table.get((state, action), 0.0)

def choose_action(state, epsilon):
    # TODO : politique epsilon-greedy : avec proba epsilon, action aleatoire (exploration)
    # sinon l'action qui maximise get_q(state, a) (exploitation)
    ...

# Boucle d'entraînement
env = gym.make('CartPole-v1')
epsilon = EPSILON_START
rewards_ql = []
epsilons_ql = []

for ep in range(N_EPISODES):
    obs, _ = env.reset()
    state = discretize(obs)
    total_reward = 0

    while True:
        action = choose_action(state, epsilon)
        obs_next, reward, terminated, truncated, _ = env.step(action)
        next_state = discretize(obs_next)
        done = terminated or truncated

        # TODO : mise a jour de Bellman (Q-learning)
        # td_target = reward + GAMMA * max_a Q(next_state, a) * (not done)
        # td_error  = td_target - Q(state, action)
        # Q(state, action) += ALPHA * td_error
        ...

        state = next_state
        total_reward += reward
        if done:
            break

    epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
    rewards_ql.append(total_reward)
    epsilons_ql.append(epsilon)

env.close()
print(f"Q-table : {len(q_table)} entrées")
print(f"Récompense moyenne (100 derniers épisodes) : {np.mean(rewards_ql[-100:]):.1f}")


### 2.3 Courbes d'apprentissage

In [ ]:
def smooth(values, window=100):
    return pd.Series(values).rolling(window, min_periods=1).mean().values

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(rewards_ql, alpha=0.2, color='steelblue')
axes[0].plot(smooth(rewards_ql), color='steelblue', linewidth=2, label='Moyenne mobile (100 ep)')
axes[0].axhline(500, color='green', linestyle='--', label='Score max (500)')
axes[0].set_xlabel('Épisode')
axes[0].set_ylabel('Récompense totale')
axes[0].set_title('Q-Learning : CartPole')
axes[0].legend()

axes[1].plot(epsilons_ql, color='orange')
axes[1].set_xlabel('Épisode')
axes[1].set_ylabel('Epsilon')
axes[1].set_title('Décroissance epsilon (exploration)')

plt.tight_layout()
plt.show()

### 2.4 Évaluation de la politique apprise

In [ ]:
def evaluate_qlearning(n_episodes=100):
    env_eval = gym.make('CartPole-v1')
    scores = []
    for _ in range(n_episodes):
        obs, _ = env_eval.reset()
        state = discretize(obs)
        total = 0
        while True:
            # TODO : choisir l'action purement gloutonne (sans exploration) selon q_table
            action = ...
            obs, reward, terminated, truncated, _ = env_eval.step(action)
            state = discretize(obs)
            total += reward
            if terminated or truncated:
                break
        scores.append(total)
    env_eval.close()
    return scores

scores_ql = evaluate_qlearning(100)
print(f"Q-Learning : CartPole (100 épisodes)")
print(f"  Moyenne : {np.mean(scores_ql):.1f}")
print(f"  Std     : {np.std(scores_ql):.1f}")
print(f"  Min/Max : {np.min(scores_ql):.0f} / {np.max(scores_ql):.0f}")


---
## 3. DQN (Deep Q-Network) : CartPole-v1

DQN remplace la Q-table par un **réseau de neurones** qui approxime $Q(s, a)$.  
Deux innovations clés par rapport au Q-Learning tabulaire :

- **Experience Replay** : les transitions $(s, a, r, s')$ sont stockées dans un buffer et réutilisées de façon aléatoire → brise les corrélations temporelles
- **Target Network** : un second réseau figé fournit les cibles $y = r + \gamma \max_{a'} Q_{\text{target}}(s', a')$ → stabilise l'apprentissage

On utilise **Stable-Baselines3** qui fournit une implémentation robuste et configurable.

### 3.1 Entraînement DQN

In [ ]:
import os
os.makedirs('logs/dqn', exist_ok=True)

env_dqn = Monitor(gym.make('CartPole-v1'), filename='logs/dqn/monitor')

# TODO : instancier un DQN('MlpPolicy', env_dqn, ...) avec les hyperparametres suivants :
# learning_rate=1e-3, buffer_size=50_000, learning_starts=1_000, batch_size=64, gamma=0.99,
# target_update_interval=500, exploration_fraction=0.2, exploration_final_eps=0.05,
# verbose=0, seed=RANDOM_STATE
model_dqn = ...

print("Architecture du réseau DQN :")
print(model_dqn.policy)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model_dqn.policy.parameters()):,}")


In [ ]:
model_dqn.learn(total_timesteps=100_000, progress_bar=True)
env_dqn.close()
print("Entraînement DQN terminé.")

### 3.2 Évaluation DQN

In [ ]:
env_eval_dqn = gym.make('CartPole-v1')
mean_dqn, std_dqn = evaluate_policy(model_dqn, env_eval_dqn, n_eval_episodes=100, deterministic=True)
env_eval_dqn.close()

print(f"DQN : CartPole (100 épisodes)")
print(f"  Moyenne : {mean_dqn:.1f}")
print(f"  Std     : {std_dqn:.1f}")

### 3.3 Courbe d'apprentissage DQN (depuis les logs Monitor)

In [ ]:
from stable_baselines3.common.results_plotter import load_results

results_dqn = load_results('logs/dqn')

plt.figure(figsize=(9, 4))
plt.plot(results_dqn['t'], results_dqn['r'], alpha=0.2, color='darkorange')
plt.plot(results_dqn['t'], smooth(results_dqn['r'], 50), color='darkorange', linewidth=2, label='Moyenne mobile')
plt.axhline(500, color='green', linestyle='--', label='Score max (500)')
plt.xlabel('Timesteps')
plt.ylabel('Récompense par épisode')
plt.title('DQN : CartPole')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. PPO (Proximal Policy Optimization) : MountainCar-v0

PPO est un algorithme **policy-based** : il optimise directement la politique $\pi(a|s)$ plutôt qu'une Q-function.

Idée clé : limiter la mise à jour de la politique à chaque step pour éviter les dégradations catastrophiques :
$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

**Pourquoi PPO sur MountainCar ?**  
MountainCar est un problème d'**exploration difficile** : la récompense est toujours -1 jusqu'au succès. PPO avec plusieurs workers parallèles explore plus efficacement que DQN.

### 4.1 Entraînement PPO

In [ ]:
os.makedirs('logs/ppo', exist_ok=True)

env_ppo = Monitor(gym.make('MountainCar-v0'), filename='logs/ppo/monitor')

# TODO : instancier un PPO('MlpPolicy', env_ppo, ...) avec les hyperparametres suivants :
# learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10, gamma=0.99,
# gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, verbose=0, seed=RANDOM_STATE
model_ppo = ...

print("Architecture du réseau PPO (actor-critic) :")
print(model_ppo.policy)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model_ppo.policy.parameters()):,}")


In [ ]:
model_ppo.learn(total_timesteps=200_000, progress_bar=True)
env_ppo.close()
print("Entraînement PPO terminé.")

### 4.2 Évaluation PPO

In [ ]:
env_eval_ppo = gym.make('MountainCar-v0')
mean_ppo, std_ppo = evaluate_policy(model_ppo, env_eval_ppo, n_eval_episodes=100, deterministic=True)
env_eval_ppo.close()

print(f"PPO : MountainCar (100 épisodes)")
print(f"  Moyenne : {mean_ppo:.1f}  (max théorique : -100 si atteint en 100 steps)")
print(f"  Std     : {std_ppo:.1f}")
print(f"  Score baseline (random) : ~-200")

### 4.3 Courbe d'apprentissage PPO

In [ ]:
results_ppo = load_results('logs/ppo')

plt.figure(figsize=(9, 4))
plt.plot(results_ppo['t'], results_ppo['r'], alpha=0.2, color='seagreen')
plt.plot(results_ppo['t'], smooth(results_ppo['r'], 50), color='seagreen', linewidth=2, label='Moyenne mobile')
plt.axhline(-200, color='red', linestyle='--', label='Baseline random (-200)')
plt.axhline(-100, color='green', linestyle='--', label='Objectif (-100)')
plt.xlabel('Timesteps')
plt.ylabel('Récompense par épisode')
plt.title('PPO : MountainCar')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. Optimisation des hyperparamètres

On explore l'impact de quelques hyperparamètres clés sur les performances.

### 5.1 Q-Learning : impact de alpha et gamma

In [ ]:
def train_qlearning(alpha, gamma, n_episodes=2000, epsilon_decay=0.995):
    env = gym.make('CartPole-v1')
    q = {}
    epsilon = 1.0
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        state = discretize(obs)
        total = 0
        while True:
            # TODO : meme politique epsilon-greedy et mise a jour de Bellman que dans la section 2
            if random.random() < epsilon:
                action = ...
            else:
                action = ...
            obs_n, r, term, trunc, _ = env.step(action)
            ns = discretize(obs_n)
            done = term or trunc
            best = ...
            q[(state, action)] = ...
            state = ns
            total += r
            if done:
                break
        epsilon = max(0.01, epsilon * epsilon_decay)
        rewards.append(total)
    env.close()
    return np.mean(rewards[-200:])

alphas = [0.05, 0.1, 0.3, 0.5]
gammas = [0.90, 0.95, 0.99]

results_hp = {}
for a in alphas:
    for g in gammas:
        score = train_qlearning(alpha=a, gamma=g)
        results_hp[(a, g)] = score
        print(f"alpha={a}, gamma={g} → {score:.1f}")

# Heatmap
hp_matrix = np.array([[results_hp[(a, g)] for g in gammas] for a in alphas])
plt.figure(figsize=(7, 5))
sns.heatmap(hp_matrix, annot=True, fmt='.0f', xticklabels=gammas, yticklabels=alphas,
            cmap='YlGnBu', linewidths=0.5)
plt.xlabel('Gamma (discount)')
plt.ylabel('Alpha (learning rate)')
plt.title('Q-Learning : Récompense moyenne (200 derniers épisodes)')
plt.tight_layout()
plt.show()


### 5.2 DQN : impact du learning rate

In [ ]:
lr_values = [1e-4, 5e-4, 1e-3, 5e-3]
scores_lr = {}

for lr in lr_values:
    env_tmp = gym.make('CartPole-v1')
    # TODO : instancier et entrainer un DQN avec ce learning_rate (50_000 timesteps), puis evaluer sur 50 episodes
    # Indice : DQN('MlpPolicy', env_tmp, learning_rate=lr, buffer_size=50_000, learning_starts=1_000,
    #              batch_size=64, gamma=0.99, exploration_fraction=0.2, verbose=0, seed=RANDOM_STATE)
    m = ...
    m.learn(total_timesteps=50_000)
    mean_r, _ = evaluate_policy(m, env_tmp, n_eval_episodes=50, deterministic=True)
    scores_lr[lr] = mean_r
    env_tmp.close()
    print(f"lr={lr:.0e} → {mean_r:.1f}")

plt.figure(figsize=(7, 4))
plt.bar([str(lr) for lr in lr_values], scores_lr.values(), color='darkorange', edgecolor='k')
plt.axhline(500, color='green', linestyle='--', label='Score max')
plt.xlabel('Learning rate')
plt.ylabel('Récompense moyenne (50 épisodes)')
plt.title('DQN : Impact du learning rate')
plt.legend()
plt.tight_layout()
plt.show()


---
## 6. Comparaison des algorithmes

### 6.1 Tableau récapitulatif

In [ ]:
# Récupération des scores DQN sous forme de liste
env_eval_dqn2 = gym.make('CartPole-v1')
scores_dqn_list = []
for _ in range(100):
    obs, _ = env_eval_dqn2.reset()
    total = 0
    while True:
        action, _ = model_dqn.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env_eval_dqn2.step(action)
        total += r
        if term or trunc:
            break
    scores_dqn_list.append(total)
env_eval_dqn2.close()

# Récupération des scores PPO sous forme de liste
env_eval_ppo2 = gym.make('MountainCar-v0')
scores_ppo_list = []
for _ in range(100):
    obs, _ = env_eval_ppo2.reset()
    total = 0
    while True:
        action, _ = model_ppo.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env_eval_ppo2.step(action)
        total += r
        if term or trunc:
            break
    scores_ppo_list.append(total)
env_eval_ppo2.close()

summary = pd.DataFrame([
    {'Algo': 'Q-Learning', 'Env': 'CartPole-v1', 'Moyenne': np.mean(scores_ql), 'Std': np.std(scores_ql),
     'Min': np.min(scores_ql), 'Max': np.max(scores_ql), 'Score max possible': 500},
    {'Algo': 'DQN',        'Env': 'CartPole-v1', 'Moyenne': np.mean(scores_dqn_list), 'Std': np.std(scores_dqn_list),
     'Min': np.min(scores_dqn_list), 'Max': np.max(scores_dqn_list), 'Score max possible': 500},
    {'Algo': 'PPO',        'Env': 'MountainCar-v0', 'Moyenne': np.mean(scores_ppo_list), 'Std': np.std(scores_ppo_list),
     'Min': np.min(scores_ppo_list), 'Max': np.max(scores_ppo_list), 'Score max possible': -100},
])
summary.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# CartPole : Q-Learning vs DQN
axes[0].hist(scores_ql, bins=20, alpha=0.6, color='steelblue', label=f'Q-Learning (moy={np.mean(scores_ql):.0f})', edgecolor='k')
axes[0].hist(scores_dqn_list, bins=20, alpha=0.6, color='darkorange', label=f'DQN (moy={np.mean(scores_dqn_list):.0f})', edgecolor='k')
axes[0].axvline(500, color='green', linestyle='--', label='Score max')
axes[0].set_xlabel('Récompense')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('CartPole : Q-Learning vs DQN')
axes[0].legend()

# MountainCar : PPO
axes[1].hist(scores_ppo_list, bins=20, alpha=0.7, color='seagreen', edgecolor='k',
             label=f'PPO (moy={np.mean(scores_ppo_list):.0f})')
axes[1].axvline(-200, color='red', linestyle='--', label='Baseline random')
axes[1].axvline(-100, color='green', linestyle='--', label='Objectif')
axes[1].set_xlabel('Récompense')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('MountainCar : PPO')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Conclusion

| Critère | Q-Learning | DQN | PPO |
|---|---|---|---|
| Famille | Tabulaire | Value-based (deep) | Policy-based (deep) |
| Espace d'états | Discret (discrétisé) | Continu | Continu |
| Espace d'actions | Discret | Discret | Discret / Continu |
| Réseau de neurones | Non | Oui | Oui (actor-critic) |
| Replay buffer | Non | Oui | Non (on-policy) |
| Scalabilité | Faible (explosion Q-table) | Bonne | Très bonne |
| Stabilité | Fragile | Moyenne | Robuste |
| Cas d'usage | Pédagogique, espaces simples | Jeux Atari, espaces moyens | Robotique, environnements complexes |

**À retenir :**
- Q-Learning est le fondement théorique : indispensable pour comprendre les autres
- DQN résout le problème de scalabilité mais reste off-policy et sensible aux hyperparamètres
- PPO est le standard industriel actuel : stable, efficace, applicable à presque tout

**Pistes d'exploration :**
- Double DQN (réduire l'overestimation des Q-values)
- A2C (Advantage Actor-Critic) : version synchrone de A3C
- SAC (Soft Actor-Critic) : PPO-like pour espaces d'actions continus
- Tester PPO sur `LunarLander-v2` ou `BipedalWalker-v3`

---
## Session à rendre

Cette section est à compléter et à rendre à l'issue du TP. Réponds à chaque question dans la
cellule *Réponse* juste en dessous, à partir des résultats que **tu as toi-même obtenus** en
réalisant ce notebook (il n'y a pas de réponse générique valable pour tout le monde : les valeurs
numériques, choix d'hyperparamètres et graphiques dépendent de ton exécution).

**Q1.** Après entraînement, quelle récompense moyenne obtiens-tu avec Q-Learning sur CartPole (100 épisodes) ? Est-ce proche du score maximum (500) ?

*Réponse :*

_(à compléter)_

**Q2.** Quelle récompense moyenne obtiens-tu avec DQN sur CartPole ? Comment se compare-t-elle à celle de Q-Learning tabulaire, et à quoi attribues-tu la différence ?

*Réponse :*

_(à compléter)_

**Q3.** Quelle récompense moyenne obtiens-tu avec PPO sur MountainCar ? Comment se compare-t-elle à la baseline aléatoire (~-200) et à l'objectif (-100) ?

*Réponse :*

_(à compléter)_

**Q4.** D'après ta heatmap alpha × gamma pour Q-Learning, quelle combinaison donne les meilleurs résultats ? Comment interprètes-tu l'effet de gamma (discount) sur ce problème ?

*Réponse :*

_(à compléter)_

**Q5.** D'après ton graphique d'impact du learning rate sur DQN, quelle valeur donne les meilleurs résultats ? Que se passe-t-il visiblement pour les learning rates trop élevés ou trop faibles ?

*Réponse :*

_(à compléter)_

**Q6.** En te basant sur ton tableau comparatif final, explique en 3-4 phrases pourquoi Q-Learning, DQN et PPO ne sont pas interchangeables : sur quel critère chacun l'emporte-t-il ?

*Réponse :*

_(à compléter)_